# rm-optpessimal-personas — Colab runner

Runs the GPU-dependent generation scripts from [puffables/rm-optpessimal-personas](https://github.com/puffables/rm-optpessimal-personas) on Colab compute.

**Before you start:**
1. `Runtime > Change runtime type > GPU > A100`. Everything here also runs on a T4/L4, but A100 cuts wall-clock time substantially — worth it if you're optimizing for speed over compute-unit cost. The two 27B Gemma-2 base models still won't fit even on a 40GB A100 (they need ~54GB just for weights); Cell 6 skips them by default regardless of GPU.
2. Create a HuggingFace token at https://huggingface.co/settings/tokens (read access is enough) and accept the license on each gated model page you need (`google/gemma-*`, `meta-llama/Llama-3.2-3B-Instruct`).
3. This repo is **private**, so cloning it needs a GitHub token too: create a fine-grained personal access token at https://github.com/settings/tokens?type=beta scoped only to the `puffables/rm-optpessimal-personas` repo, with **Contents: Read-only** permission.
4. In the Colab left sidebar, click the key icon ("Secrets") and add two secrets — `HF_TOKEN` and `GH_TOKEN` — with those tokens, toggling "Notebook access" on for both. This keeps both tokens out of the notebook itself.

**Data persistence:** the repo's `data/` folder is checked into git, so a fresh clone already has prior results, and the three checkpointing scripts (everything except `generate_base_model_logprobs.py`) will skip work that's already done. The last cell zips up anything new/changed so you can pull it back into your local clone and commit it — this notebook does not push to GitHub for you.

In [1]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
# 2. Clone the repo (fresh each session)
# Private repo, so the clone is authenticated with GH_TOKEN (from Colab secrets).
# The token is only embedded in the URL for the clone itself; the remote is
# rewritten to drop it immediately after so it isn't left sitting in .git/config.
# Uses an absolute path so this cell is safe to re-run mid-session without
# nesting a clone inside itself.
import os
from google.colab import userdata

REPO_PATH = "puffables/rm-optpessimal-personas"
REPO_DIR = "/content/rm-optpessimal-personas"
GH_TOKEN = userdata.get("GH_TOKEN")

if not os.path.exists(REPO_DIR):
    !git clone https://{GH_TOKEN}@github.com/{REPO_PATH}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin kv-cached
!git checkout kv-cached
!git remote set-url origin https://github.com/{REPO_PATH}.git

Cloning into '/content/rm-optpessimal-personas'...
remote: Enumerating objects: 262, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 262 (delta 8), reused 55 (delta 7), pack-reused 195 (from 1)
Receiving objects: 100% (262/262), 185.54 MiB | 28.00 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (140/140), done.
/content/rm-optpessimal-personas
From https://github.com/puffables/rm-optpessimal-personas
 * branch            kv-cached  -> FETCH_HEAD
Branch 'kv-cached' set up to track remote branch 'kv-cached' from 'origin'.
Switched to a new branch 'kv-cached'


In [3]:
# 3. Install requirements
# Colab's preinstalled torch is already CUDA-enabled, so this mainly adds
# transformers (>=4.45, for the rope_scaling fix Llama-3.x needs), accelerate,
# and the smaller deps.
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
# 4. HuggingFace auth (reads the HF_TOKEN secret set up in the sidebar)
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [6]:
# 5. Sanity-check gated model access before burning compute on a script that'll 401 midway through
from huggingface_hub import model_info

for m in ["google/gemma-2b", "google/gemma-2-9b", "google/gemma-2-27b", "meta-llama/Llama-3.2-3B-Instruct"]:
    try:
        model_info(m)
        print(f"OK   {m}")
    except Exception as e:
        print(f"FAIL {m}: {e}")

OK   google/gemma-2b
OK   google/gemma-2-9b
OK   google/gemma-2-27b
OK   meta-llama/Llama-3.2-3B-Instruct


## 6. Reward model scores
By default `config/reward_models.yaml` has only one *active* (uncommented) model — `Ray2333/GRM-Llama3.2-3B-rewardmodel-ft` (3B) — the rest are commented out. The YAML's configured batch size (128) was tuned conservatively; on an A100 you can push it much higher with `--batch-size` to cut runtime.

In [7]:
!python generate_reward_model_scores.py --batch-size 512

Skipping Ray2333/GRM-Llama3.2-3B-rewardmodel-ft — all prompts already scored
Done.


## 6b. KV-cached comparison (this branch)
The `kv-cached` branch adds a `--kv-cache` scoring path that caches the shared prompt prefix once per prompt instead of re-tokenizing and re-running the whole sequence per candidate token — much faster on a real GPU with a big batch size. Building it also surfaced two pre-existing issues in the original scoring path (duplicate BOS token, and a decode-then-re-tokenize round trip that can silently substitute a different token for a large fraction of the vocabulary), both fixed in this path by default.

This writes to a **separate output directory** — not `data/reward_model_scores` — so the checkpoint-skip logic in the script doesn't treat the prompts scored above as already done.

In [8]:
!python generate_reward_model_scores.py --batch-size 1024 --kv-cache --output-dir data/reward_model_scores_kv_cached

Processing model: Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (7 prompts to score)
Found 1 CUDA devices
tokenizer_config.json: 54.7kB [00:00, 56.6MB/s]
tokenizer.json: 9.09MB [00:00, 24.1MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 4.05MB/s]
Using batch size 128 with single GPU setup (cuda)
config.json: 1.13kB [00:00, 7.44MB/s]
2026-08-02 15:52:13.925501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-02 15:52:13.994138: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors.index.json: 21.0kB

## 6c. Diff against the baseline run

In [9]:
import pandas as pd

baseline = pd.read_csv("data/reward_model_scores/Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv")
kv_cached = pd.read_csv("data/reward_model_scores_kv_cached/Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv")

prompt_cols = [c for c in baseline.columns if c not in ("token_id", "token_name", "token_decoded")]

for col in prompt_cols:
    diff = (baseline[col] - kv_cached[col]).abs()
    baseline_top10 = set(baseline.nlargest(10, col)["token_decoded"])
    kv_top10 = set(kv_cached.nlargest(10, col)["token_decoded"])
    overlap = len(baseline_top10 & kv_top10)
    print(f"{col:25s}  mean|diff|={diff.mean():.3f}  max|diff|={diff.max():.3f}  top-10 overlap={overlap}/10")


greatest                   mean|diff|=0.444  max|diff|=6.234  top-10 overlap=8/10
best                       mean|diff|=0.449  max|diff|=5.932  top-10 overlap=5/10
worst                      mean|diff|=0.415  max|diff|=4.344  top-10 overlap=4/10
greatest_plain             mean|diff|=0.497  max|diff|=5.051  top-10 overlap=5/10
greatest_i_think           mean|diff|=0.409  max|diff|=4.648  top-10 overlap=4/10
greatest_you_think         mean|diff|=0.519  max|diff|=4.582  top-10 overlap=7/10
greatest_people_think      mean|diff|=0.394  max|diff|=4.656  top-10 overlap=4/10


## 7. Persona-conditioned reward model scores
Same active model set as above, swept over `config/personas.yaml` x `config/persona_prompts.yaml` — 227 combinations, the slowest step in this pipeline. This script's default batch size (1024) was tuned for a 97GB workstation GPU; on a 40GB A100, 384 leaves headroom for the longer persona-prefixed prompts. If you land on a smaller GPU instead, drop this back to 128 (T4) or lower.

In [ ]:
!python generate_persona_reward_model_scores.py --batch-size 384

## 8. Persona-conditioned base model logprobs
Uses `config/llama_base_models.yaml`, which lists only `meta-llama/Llama-3.2-3B-Instruct` — fits on a T4.

In [ ]:
!python generate_persona_base_model_logprobs.py

## 9. Base model logprobs (the heavy one)
`config/gemma_base_models.yaml` spans gemma-1/2 from 2B up to **27B**. Unlike the three scripts above, this one has no resume/skip logic — it recomputes and overwrites every model's CSV on every run, so only pick the models you actually need.

The two 27B variants need ~54GB just for weights in bf16, which doesn't fit even on Colab's 40GB A100 — the filtered config below (which drops them) is the right default regardless of GPU tier. Their outputs are already present in `data/base_model_logits/` from a prior run — only rerun them (on hardware that can actually hold them) if `config/prompts.yaml` has changed since.

In [ ]:
# Build a Colab-friendly config that excludes the 27B models (not committed to the repo)
import yaml

with open("config/gemma_base_models.yaml") as f:
    all_models = yaml.safe_load(f)

small_models = [m for m in all_models if "27b" not in m["name"]]
with open("config/gemma_base_models_no27b.yaml", "w") as f:
    yaml.safe_dump(small_models, f)

print(f"{len(small_models)}/{len(all_models)} models kept:")
for m in small_models:
    print(" ", m["name"])

In [ ]:
!python generate_base_model_logprobs.py --config gemma_base_models_no27b.yaml

# Full sweep including the 27B models — only run this on an A100 80GB:
# !python generate_base_model_logprobs.py --config gemma_base_models.yaml

## 10. Pull results back out
Zips the `data/` outputs so you can download them and merge into your local clone (`git status` there will show only the files that actually changed).

In [11]:
from google.colab import files

!zip -qr data_outputs.zip data/reward_model_scores data/reward_model_scores_kv_cached data/persona_reward_model_scores data/persona_base_model_logits data/base_model_logits
files.download("data_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>